# 07 — Prompt iteration: `backward_v1` (LLM backward, Method D)

Purpose: identical structure to `06_prompt_iteration.ipynb`, but for
Method D — run `backward_v1` on the **same 20** Spider train examples the forward phase
used (`data/processed/prompt_iteration_set.json`, `seed=42`), so C and D are
directly comparable on identical questions. Score against **sqlglot Tier-2
train gold** (`gold_links_train_all_sql.json`) — Method D's SQL-derived
predictions are inherently Tier-2-shaped (join columns included by
construction), unlike Method C which is prompted to exclude them.

**This notebook makes real calls to the Anthropic API** (Haiku 4.5), bounded
by `cost_cap_usd = cumulative_cap + 3` where `cumulative_cap` is whatever is
already logged in the shared `outputs/logs/llm_calls_prompt_iteration.jsonl`
(same file the forward phase used — this notebook's phase is tagged
`"backward_prompt_iteration"` for traceability within it). Never run on
Spider dev here.

Constraints (locked before running):
- Same 20 `question_id`s as the forward phase — no resampling.
- Do not tune the prompt to fix specific examples — only general rule
  changes are in scope for a `backward_v2` draft.
- Method D is deterministic (`k_samples=1`, `temperature=0.0`) — no
  self-consistency sampling, so (unlike the forward phase) there is no run-to-run
  noise to average over.


In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))


def load_dotenv(path: Path) -> None:
    """Minimal .env loader (no python-dotenv dependency for one env var)."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in ("'", '"'):
            value = value[1:-1]
        os.environ.setdefault(key, value)


load_dotenv(REPO_ROOT / ".env")

from schema_linking.base import from_predictions_to_dict
from schema_linking.data_loader import load_spider_questions
from schema_linking.evaluator import evaluate
from schema_linking.llm_linker import LLMBackwardLinker, LLMForwardLinker
from schema_linking.schema_parser import load_schemas
from schema_linking.utils.difficulty import difficulty_for_examples
from schema_linking.utils.llm_client import LLMClient
from schema_linking.utils.prompts import (
    BACKWARD_V1,
    FORWARD_V1,
    render_backward_user_message,
    render_schema_block,
)

LOG_PATH = REPO_ROOT / "outputs" / "logs" / "llm_calls_prompt_iteration.jsonl"
SELECTION_PATH = REPO_ROOT / "data" / "processed" / "prompt_iteration_set.json"
BACKWARD_FEWSHOT_PATH = REPO_ROOT / "data" / "processed" / "few_shot_examples_backward.json"
FORWARD_FEWSHOT_PATH = REPO_ROOT / "data" / "processed" / "few_shot_examples.json"
GOLD_TRAIN_TIER1_PATH = REPO_ROOT / "data" / "processed" / "gold_links_train_mentioned.json"
GOLD_TRAIN_TIER2_PATH = REPO_ROOT / "data" / "processed" / "gold_links_train_all_sql.json"

pd


<module 'pandas' from '/Users/mac/Documents/Masters BHT 2023-2025/Semester 6 2026 april-sep/code/.venv/lib/python3.12/site-packages/pandas/__init__.py'>

## 1. Load train examples, schemas, Tier-1 + Tier-2 train gold, hardness, the fixed 20-example selection

In [2]:
train_examples = list(load_spider_questions("train"))
schemas = load_schemas()
hardness = difficulty_for_examples(train_examples)

tier1_raw = json.load(GOLD_TRAIN_TIER1_PATH.open())
gold_tier1 = {int(qid): entry for qid, entry in tier1_raw.items()}
tier2_raw = json.load(GOLD_TRAIN_TIER2_PATH.open())
gold_tier2 = {int(qid): entry for qid, entry in tier2_raw.items()}

selection_records = json.load(SELECTION_PATH.open())
selected_qids = [r["question_id"] for r in selection_records]
examples_by_qid = {ex.question_id: ex for ex in train_examples}
selected = [examples_by_qid[qid] for qid in selected_qids]

print(f"{len(train_examples)} train examples, {len(schemas)} schemas, "
      f"{len(gold_tier1)} Tier-1 gold entries, {len(gold_tier2)} Tier-2 gold entries")
print(f"Reusing the {len(selected)} selected examples (same seed=42 set, no resampling)")
pd.DataFrame(selection_records)[["question_id", "db_id", "hardness", "question"]]


7000 train examples, 166 schemas, 7000 Tier-1 gold entries, 7000 Tier-2 gold entries
Reusing the 20 selected examples (same seed=42 set, no resampling)


,question_id,db_id,hardness,question
0,1550,customers_and_invoices,easy,Count the number of customers who have an acco...
1,4052,student_1,easy,Find the last names of teachers teaching in cl...
2,6623,driving_school,easy,What are the ids of all vehicles?
3,235,musical,easy,Count the number of actors.
4,452,allergy_1,easy,How many animal type allergies exist?
5,324,product_catalog,medium,Which catalog content has the highest height? ...
6,963,medicine_enzyme_interaction,medium,What is the id and trade name of the medicines...
7,3128,assets_maintenance,medium,How many assets does each third party company ...
8,2054,party_people,medium,Which minister left office the latest?
9,5546,products_gen_characteristics,medium,What is the color code and description of the ...


## 2. Load backward few-shots (enrich with schema_block) and forward few-shots (for the bonus comparison re-run)

In [3]:
backward_few_shot = json.load(BACKWARD_FEWSHOT_PATH.open())
for ex in backward_few_shot:
    ex["schema_block"] = render_schema_block(schemas[ex["db_id"]])

forward_few_shot = json.load(FORWARD_FEWSHOT_PATH.open())
for ex in forward_few_shot:
    ex["schema_block"] = render_schema_block(schemas[ex["db_id"]])

for ex in backward_few_shot:
    print(f"{ex['pattern']:>12}: qid={ex['question_id']} db={ex['db_id']!r} — {ex['question']!r}")
    print(f"             gold_sql={ex['gold_sql']!r}")


      simple: qid=4913 db='store_product' — 'What is the total number of residents for the districts with the 3 largest areas?'
             gold_sql='SELECT sum(city_population) FROM district ORDER BY city_area DESC LIMIT 3'
 multi_table: qid=334 db='product_catalog' — 'Which attribute definitions have attribute value 0? Give me the attribute name and attribute ID.'
             gold_sql='SELECT t1.attribute_name ,  t1.attribute_id FROM Attribute_Definitions AS t1 JOIN Catalog_Contents_Additional_Attributes AS t2 ON t1.attribute_id  =  t2.attribute_id WHERE t2.attribute_value  =  0'


## 3. Compute cumulative_cap from the shared iteration-phase log

In [4]:
cumulative_cap = 0.0
if LOG_PATH.exists():
    with LOG_PATH.open() as f:
        for line in f:
            line = line.strip()
            if line:
                cumulative_cap += json.loads(line)["cost_usd"]

cost_cap_usd = cumulative_cap + 3.0
print(f"Already logged in {LOG_PATH.name}: ${cumulative_cap:.5f}")
print(f"This run's cost_cap_usd: ${cost_cap_usd:.5f}")


Already logged in llm_calls_prompt_iteration.jsonl: $0.25200
This run's cost_cap_usd: $3.25201


## 4. Run backward_v1 on the 20 examples (k=1, T=0)

In [5]:
llm_client_backward_v1 = LLMClient(
    model="claude-haiku-4-5-20251001",
    temperature=0.0,
    max_tokens=1024,
    log_path=LOG_PATH,
    cost_cap_usd=cost_cap_usd,
)
linker_backward_v1 = LLMBackwardLinker(
    llm_client=llm_client_backward_v1,
    prompt=BACKWARD_V1,
    few_shot=backward_few_shot,
    extra_metadata={"phase": "backward_prompt_iteration", "prompt_version": BACKWARD_V1.version},
)

predictions_backward_v1 = {}
for i, ex in enumerate(selected):
    schema = schemas[ex.db_id]
    pred = linker_backward_v1.predict_one(ex, schema)
    predictions_backward_v1[ex.question_id] = pred
    t1 = gold_tier1[ex.question_id]
    t2 = gold_tier2[ex.question_id]
    print("=" * 100)
    print(f"[{i + 1:>2}/20] qid={ex.question_id:<5} db={ex.db_id:<28} hardness={hardness[ex.question_id]:<6} "
          f"cost=${pred.extra['cost_usd']:.5f}")
    print(f"Question       : {ex.question}")
    print(f"Gold SQL        : {ex.query}")
    print(f"LLM SQL         : {pred.extra['raw_sql']}")
    print(f"Parsed tables   : {list(pred.tables)}")
    print(f"Parsed columns  : {[list(c) for c in pred.columns]}")
    print(f"Gold Tier-1     : tables={t1['tables']} columns={t1['columns']}")
    print(f"Gold Tier-2     : tables={t2['tables']} columns={t2['columns']}")
    print(f"Parse issues    : {pred.extra['parse_issues']}")

print(f"\nbackward_v1 total cost: ${sum(p.extra['cost_usd'] for p in predictions_backward_v1.values()):.5f}")


[ 1/20] qid=1550  db=customers_and_invoices       hardness=easy   cost=$0.00246
Question       : Count the number of customers who have an account.
Gold SQL        : SELECT count(DISTINCT customer_id) FROM Accounts
LLM SQL         : SELECT COUNT(DISTINCT customer_id) FROM Accounts
Parsed tables   : ['Accounts']
Parsed columns  : [['Accounts', 'customer_id']]
Gold Tier-1     : tables=['Accounts'] columns=[['Accounts', 'customer_id']]
Gold Tier-2     : tables=['Accounts'] columns=[['Accounts', 'customer_id']]
Parse issues    : []


[ 2/20] qid=4052  db=student_1                    hardness=easy   cost=$0.00110
Question       : Find the last names of teachers teaching in classroom 109.
Gold SQL        : SELECT lastname FROM teachers WHERE classroom  =  109
LLM SQL         : SELECT LastName FROM teachers WHERE Classroom = 109
Parsed tables   : ['teachers']
Parsed columns  : [['teachers', 'Classroom'], ['teachers', 'LastName']]
Gold Tier-1     : tables=['teachers'] columns=[['teachers', 'Classroom'], ['teachers', 'LastName']]
Gold Tier-2     : tables=['teachers'] columns=[['teachers', 'Classroom'], ['teachers', 'LastName']]
Parse issues    : []


[ 3/20] qid=6623  db=driving_school               hardness=easy   cost=$0.00192
Question       : What are the ids of all vehicles?
Gold SQL        : SELECT vehicle_id FROM Vehicles;
LLM SQL         : SELECT vehicle_id FROM Vehicles
Parsed tables   : ['Vehicles']
Parsed columns  : [['Vehicles', 'vehicle_id']]
Gold Tier-1     : tables=['Vehicles'] columns=[['Vehicles', 'vehicle_id']]
Gold Tier-2     : tables=['Vehicles'] columns=[['Vehicles', 'vehicle_id']]
Parse issues    : []


[ 4/20] qid=235   db=musical                      hardness=easy   cost=$0.00117
Question       : Count the number of actors.
Gold SQL        : SELECT count(*) FROM actor
LLM SQL         : SELECT COUNT(*) FROM actor
Parsed tables   : ['actor']
Parsed columns  : []
Gold Tier-1     : tables=['actor'] columns=[]
Gold Tier-2     : tables=['actor'] columns=[]
Parse issues    : []


[ 5/20] qid=452   db=allergy_1                    hardness=easy   cost=$0.00132
Question       : How many animal type allergies exist?
Gold SQL        : SELECT count(*) FROM Allergy_type WHERE allergytype  =  "animal"
LLM SQL         : SELECT count(*) FROM Allergy_Type WHERE AllergyType = 'animal'
Parsed tables   : ['Allergy_Type']
Parsed columns  : [['Allergy_Type', 'AllergyType']]
Gold Tier-1     : tables=['Allergy_Type'] columns=[['Allergy_Type', 'AllergyType']]
Gold Tier-2     : tables=['Allergy_Type'] columns=[['Allergy_Type', 'AllergyType']]
Parse issues    : []


...

[16/20] qid=4610  db=entertainment_awards         hardness=hard   cost=$0.00152
Question       : Show the names of festivals that have nominated artworks of type "Program Talent Show".
Gold SQL        : SELECT T3.Festival_Name FROM nomination AS T1 JOIN artwork AS T2 ON T1.Artwork_ID  =  T2.Artwork_ID JOIN festival_detail AS T3 ON T1.Festival_ID  =  T3.Festival_ID WHERE T2.Type  =  "Program Talent Show"
LLM SQL         : SELECT DISTINCT f.Festival_Name FROM festival_detail f JOIN nomination n ON f.Festival_ID = n.Festival_ID JOIN artwork a ON n.Artwork_ID = a.Artwork_ID WHERE a.Type = 'Program Talent Show'
Parsed tables   : ['artwork', 'festival_detail', 'nomination']
Parsed columns  : [['artwork', 'Artwork_ID'], ['artwork', 'Type'], ['festival_detail', 'Festival_ID'], ['festival_detail', 'Festival_Name'], ['nomination', 'Artwork_ID'], ['nomination', 'Festival_ID']]
Gold Tier-1     : tables=['artwork', 'festival_detail'] columns=[['artwork', 'Type'], ['festival_detail', 'Festival_Name'

[17/20] qid=3385  db=sports_competition           hardness=hard   cost=$0.00178
Question       : which countries did participated in both Friendly and Tournament type competitions.
Gold SQL        : SELECT country FROM competition WHERE competition_type  =  'Friendly' INTERSECT SELECT country FROM competition WHERE competition_type  =  'Tournament'
LLM SQL         : SELECT DISTINCT c.Country
FROM competition c
WHERE c.Competition_type = 'Friendly'
AND c.Country IN (
  SELECT Country
  FROM competition
  WHERE Competition_type = 'Tournament'
)
Parsed tables   : ['competition']
Parsed columns  : [['competition', 'Competition_type'], ['competition', 'Country']]
Gold Tier-1     : tables=['competition'] columns=[['competition', 'Competition_type'], ['competition', 'Country']]
Gold Tier-2     : tables=['competition'] columns=[['competition', 'Competition_type'], ['competition', 'Country']]
Parse issues    : []


[18/20] qid=5877  db=tracking_share_transactions  hardness=extra  cost=$0.00191
Question       : Show the description of the transaction type that occurs most frequently.
Gold SQL        : SELECT T1.transaction_type_description FROM Ref_Transaction_Types AS T1 JOIN TRANSACTIONS AS T2 ON T1.transaction_type_code  =  T2.transaction_type_code GROUP BY T1.transaction_type_code ORDER BY COUNT(*) DESC LIMIT 1
LLM SQL         : SELECT transaction_type_description FROM Ref_Transaction_Types WHERE transaction_type_code = (SELECT transaction_type_code FROM Transactions GROUP BY transaction_type_code ORDER BY COUNT(*) DESC LIMIT 1)
Parsed tables   : ['Ref_Transaction_Types', 'Transactions']
Parsed columns  : [['Ref_Transaction_Types', 'transaction_type_code'], ['Ref_Transaction_Types', 'transaction_type_description']]
Gold Tier-1     : tables=['Ref_Transaction_Types'] columns=[['Ref_Transaction_Types', 'transaction_type_code'], ['Ref_Transaction_Types', 'transaction_type_description']]
Gold Tier-

[19/20] qid=5794  db=customer_complaints          hardness=extra  cost=$0.00188
Question       : Find the last name of the staff member who processed the complaint of the cheapest product.
Gold SQL        : SELECT t1.last_name FROM staff AS t1 JOIN complaints AS t2 ON t1.staff_id  =  t2.staff_id JOIN products AS t3 ON t2.product_id  =  t3.product_id ORDER BY t3.product_price LIMIT 1
LLM SQL         : SELECT s.last_name FROM Staff s JOIN Complaints c ON s.staff_id = c.staff_id JOIN Products p ON c.product_id = p.product_id WHERE p.product_price = (SELECT MIN(product_price) FROM Products)
Parsed tables   : ['Complaints', 'Products', 'Staff']
Parsed columns  : [['Complaints', 'product_id'], ['Complaints', 'staff_id'], ['Products', 'product_id'], ['Products', 'product_price'], ['Staff', 'last_name'], ['Staff', 'staff_id']]
Gold Tier-1     : tables=['Products', 'Staff'] columns=[['Products', 'product_price'], ['Staff', 'last_name']]
Gold Tier-2     : tables=['Complaints', 'Products', 'Staff

[20/20] qid=3641  db=baseball_1                   hardness=extra  cost=$0.00641
Question       : In 2014, what are the id and rank of the team that has the largest average number of attendance?
Gold SQL        : SELECT T2.team_id ,  T2.rank FROM home_game AS T1 JOIN team AS T2 ON T1.team_id  =  T2.team_id WHERE T1.year  =  2014 GROUP BY T1.team_id ORDER BY avg(T1.attendance) DESC LIMIT 1;
LLM SQL         : SELECT team_id, rank FROM team WHERE year = 2014 ORDER BY attendance DESC LIMIT 1
Parsed tables   : ['team']
Parsed columns  : [['team', 'attendance'], ['team', 'rank'], ['team', 'team_id'], ['team', 'year']]
Gold Tier-1     : tables=['home_game', 'team'] columns=[['home_game', 'attendance'], ['home_game', 'team_id'], ['home_game', 'year'], ['team', 'rank'], ['team', 'team_id']]
Gold Tier-2     : tables=['home_game', 'team'] columns=[['home_game', 'attendance'], ['home_game', 'team_id'], ['home_game', 'year'], ['team', 'rank'], ['team', 'team_id']]
Parse issues    : []

backward_v1 t

## 5. Per-query F1 vs sqlglot Tier-2 train gold

In [6]:
def per_query_table(predictions: dict, method_name: str) -> pd.DataFrame:
    pred_dict = from_predictions_to_dict(predictions)
    gold_subset = {qid: gold_tier2[qid] for qid in pred_dict}
    result = evaluate(
        predictions=pred_dict, gold=gold_subset, schemas=schemas, hardness=hardness,
        method_name=method_name, tier_name="tier2",
    )
    per_query = result.per_query.copy()
    per_query["mean_f1"] = (per_query["table_f1"] + per_query["column_f1"]) / 2
    return per_query.sort_values("mean_f1").reset_index(drop=True)


per_query_backward_v1 = per_query_table(predictions_backward_v1, "llm_backward_v1")
per_query_backward_v1[["question_id", "db_id", "hardness", "table_f1", "column_f1", "mean_f1",
                        "hallucinated_tables_list", "hallucinated_columns_list"]]


,question_id,db_id,hardness,table_f1,column_f1,mean_f1,hallucinated_tables_list,hallucinated_columns_list
0,3641,baseball_1,extra,0.666667,0.444444,0.555556,,
1,3128,assets_maintenance,medium,1.000000,0.800000,0.900000,,
2,5877,tracking_share_transactions,extra,1.000000,0.800000,0.900000,,
3,963,medicine_enzyme_interaction,medium,1.000000,0.857143,0.928571,,
4,235,musical,easy,1.000000,1.000000,1.000000,,
5,5794,customer_complaints,extra,1.000000,1.000000,1.000000,,
6,5546,products_gen_characteristics,medium,1.000000,1.000000,1.000000,,
7,4915,store_product,medium,1.000000,1.000000,1.000000,,
8,4610,entertainment_awards,hard,1.000000,1.000000,1.000000,,
9,4098,company_employee,medium,1.000000,1.000000,1.000000,,


## 6. Worst 5 cases — full detail

In [7]:
worst5_qids_backward_v1 = per_query_backward_v1["question_id"].head(5).tolist()

for qid in worst5_qids_backward_v1:
    ex = examples_by_qid[qid]
    pred = predictions_backward_v1[qid]
    t1, t2 = gold_tier1[qid], gold_tier2[qid]
    row = per_query_backward_v1[per_query_backward_v1["question_id"] == qid].iloc[0]
    print("=" * 100)
    print(f"qid={qid} db={ex.db_id} hardness={hardness[qid]}")
    print(f"Question: {ex.question}")
    print(f"Gold SQL: {ex.query}")
    print(f"LLM  SQL: {pred.extra['raw_sql']}")
    print(f"Gold Tier-1: tables={t1['tables']} columns={t1['columns']}")
    print(f"Gold Tier-2: tables={t2['tables']} columns={t2['columns']}")
    print(f"Predicted  : tables={list(pred.tables)} columns={[list(c) for c in pred.columns]}")
    print(f"table_f1={row['table_f1']:.3f} column_f1={row['column_f1']:.3f}")
    print(f"Parse issues: {pred.extra['parse_issues']}")


qid=3641 db=baseball_1 hardness=extra
Question: In 2014, what are the id and rank of the team that has the largest average number of attendance?
Gold SQL: SELECT T2.team_id ,  T2.rank FROM home_game AS T1 JOIN team AS T2 ON T1.team_id  =  T2.team_id WHERE T1.year  =  2014 GROUP BY T1.team_id ORDER BY avg(T1.attendance) DESC LIMIT 1;
LLM  SQL: SELECT team_id, rank FROM team WHERE year = 2014 ORDER BY attendance DESC LIMIT 1
Gold Tier-1: tables=['home_game', 'team'] columns=[['home_game', 'attendance'], ['home_game', 'team_id'], ['home_game', 'year'], ['team', 'rank'], ['team', 'team_id']]
Gold Tier-2: tables=['home_game', 'team'] columns=[['home_game', 'attendance'], ['home_game', 'team_id'], ['home_game', 'year'], ['team', 'rank'], ['team', 'team_id']]
Predicted  : tables=['team'] columns=[['team', 'attendance'], ['team', 'rank'], ['team', 'team_id'], ['team', 'year']]
table_f1=0.667 column_f1=0.444
Parse issues: []
qid=3128 db=assets_maintenance hardness=medium
Question: How many asse

## 7. Failure categorisation (programmatic, fixed vocabulary)

In [8]:
CATEGORIES = [
    "Parse error",
    "Hallucinated table",
    "Hallucinated column",
    "Missed a required table",
    "Missed a required column",
    "Excess join column",  # Tier-1-only failure — expected, not a real Tier-2 failure
    "Other",
]


def categorize(qid: int, pred, t1: dict, t2: dict) -> str:
    issues = pred.extra["parse_issues"]
    if any(i["kind"] == "parse_error" for i in issues):
        return "Parse error"
    if any(i["kind"] == "unknown_table" for i in issues):
        return "Hallucinated table"
    if any(i["kind"] == "unknown_column" for i in issues):
        return "Hallucinated column"

    pred_tables = set(pred.tables)
    pred_columns = {tuple(c) for c in pred.columns}
    t2_tables = set(t2["tables"])
    t2_columns = {tuple(c) for c in t2["columns"]}
    t1_columns = {tuple(c) for c in t1["columns"]}

    if t2_tables - pred_tables:
        return "Missed a required table"
    if t2_columns - pred_columns:
        return "Missed a required column"
    if pred_columns & (t2_columns - t1_columns):
        return "Excess join column"
    return "Other"


categories_all20 = {
    qid: categorize(qid, predictions_backward_v1[qid], gold_tier1[qid], gold_tier2[qid])
    for qid in selected_qids
}
tally_all20 = pd.Series(categories_all20).value_counts()
print("Tally across all 20 examples:")
print(tally_all20)

categories_worst5 = {qid: categories_all20[qid] for qid in worst5_qids_backward_v1}
tally_worst5 = pd.Series(categories_worst5).value_counts()
print("\nTally across the worst 5 (the decision-rule population):")
print(tally_worst5)
for qid, cat in categories_worst5.items():
    print(f"  qid={qid}: {cat}")


Tally across all 20 examples:
Other                       12
Excess join column           6
Missed a required column     1
Missed a required table      1
Name: count, dtype: int64

Tally across the worst 5 (the decision-rule population):
Excess join column          2
Missed a required table     1
Missed a required column    1
Other                       1
Name: count, dtype: int64
  qid=3641: Missed a required table
  qid=3128: Excess join column
  qid=5877: Missed a required column
  qid=963: Excess join column
  qid=235: Other


## 8. STOP — tally and decision

In [9]:
dominant_category = tally_worst5.idxmax()
dominant_count = tally_worst5.max()
meets_threshold = dominant_count >= 3 and dominant_category not in ("Other", "Excess join column")

print(f"Dominant category among worst 5: {dominant_category!r} ({dominant_count}/5)")
print(f"Meets '>=3 share a fixable category' rule: {meets_threshold}")
print("(Excess join column is expected/not fixable by definition — Tier-2 scoring already "
      "credits it; Other has no actionable rule change by definition.)")


Dominant category among worst 5: 'Excess join column' (2/5)
Meets '>=3 share a fixable category' rule: False
(Excess join column is expected/not fixable by definition — Tier-2 scoring already credits it; Other has no actionable rule change by definition.)


## 9. Bonus comparison — does D's predicted schema set intersect with C's, on the same 20 questions?

In [10]:
llm_client_forward_compare = LLMClient(
    model="claude-haiku-4-5-20251001",
    temperature=0.3,
    max_tokens=1024,
    log_path=LOG_PATH,
    cost_cap_usd=cost_cap_usd,
)
linker_forward_compare = LLMForwardLinker(
    llm_client=llm_client_forward_compare,
    prompt=FORWARD_V1,
    few_shot=forward_few_shot,
    k_samples=3,
    aggregation="union",
    extra_metadata={"phase": "backward_prompt_iteration", "prompt_version": FORWARD_V1.version, "purpose": "bonus_comparison"},
)

predictions_forward_compare = {}
for ex in selected:
    predictions_forward_compare[ex.question_id] = linker_forward_compare.predict_one(ex, schemas[ex.db_id])

rows = []
for qid in selected_qids:
    c_tables = set(predictions_forward_compare[qid].tables)
    d_tables = set(predictions_backward_v1[qid].tables)
    rows.append({
        "question_id": qid,
        "db_id": examples_by_qid[qid].db_id,
        "C_tables": sorted(c_tables),
        "D_tables": sorted(d_tables),
        "intersects": bool(c_tables & d_tables),
        "n_both": len(c_tables & d_tables),
        "n_C_only": len(c_tables - d_tables),
        "n_D_only": len(d_tables - c_tables),
    })
comparison_df = pd.DataFrame(rows)
print(f"Intersect on {comparison_df['intersects'].sum()}/20 questions")
comparison_df


Intersect on 20/20 questions


,question_id,db_id,C_tables,D_tables,intersects,n_both,n_C_only,n_D_only
0,1550,customers_and_invoices,"[Accounts, Customers]",[Accounts],True,1,1,0
1,4052,student_1,[teachers],[teachers],True,1,0,0
2,6623,driving_school,[Vehicles],[Vehicles],True,1,0,0
3,235,musical,[actor],[actor],True,1,0,0
4,452,allergy_1,[Allergy_Type],[Allergy_Type],True,1,0,0
5,324,product_catalog,[Catalog_Contents],[Catalog_Contents],True,1,0,0
6,963,medicine_enzyme_interaction,"[medicine, medicine_enzyme_interaction]","[medicine, medicine_enzyme_interaction]",True,2,0,0
7,3128,assets_maintenance,"[Assets, Third_Party_Companies]","[Assets, Third_Party_Companies]",True,2,0,0
8,2054,party_people,[party],[party],True,1,0,0
9,5546,products_gen_characteristics,"[Products, Ref_Colors]","[Products, Ref_Colors]",True,2,0,0


## 10. Decision: lock `backward_v1`

**Real per-query results**: 15/20 examples score a perfect `mean_f1 = 1.0`
against sqlglot Tier-2 train gold. Only 4 are genuinely imperfect (qid 3641,
3128, 5877, 963) — the programmatic worst-5 list above also included qid
235, but that query's `mean_f1` is exactly 1.0 (a tie-break artifact of
`.head(5)` on a frame with many ties at the top, not a real failure — see
its full detail in §6, predicted == gold exactly).

**Hand-verified categorisation of the 4 real failures** (the automated
classifier's "Excess join column" label needed manual correction on two of
these — its priority order stops at the first matching rule and doesn't
separately flag an *additional* extraneous column once a genuine join
column is already found):

| qid | db | Category (corrected) | Root cause |
| --- | --- | --- | --- |
| 3641 | baseball_1 | **Missed a required table** | Genuine reasoning gap — the model never joined to `home_game`, instead reading `attendance`/`year` directly off `team`. Same qid flagged for Method C's own worst-5, same underlying gap — looks like a domain-knowledge limitation orthogonal to which prompting method is used. |
| 3128 | assets_maintenance | **Other** | Model wrote `COUNT(a.asset_id)` instead of gold's `COUNT(*)` — a valid alternate SQL formulation referencing one extra real (non-hallucinated), non-gold column. The automated classifier flagged this run as "Excess join column" because the query also correctly includes the genuine join column `Assets.supplier_company_id`; the `asset_id` over-selection isn't a distinct case in the given 7-category vocabulary. |
| 5877 | tracking_share_transactions | **Missed a required column** | Model wrote a correlated subquery (`WHERE ... = (SELECT ... FROM Transactions ...)`) instead of gold's JOIN. The subquery's unqualified `transaction_type_code` should resolve to `Transactions.transaction_type_code`, but the shared `extract_schema_references` walker's ambiguous-unqualified-column heuristic (documented as a known false-negative source) attributes it elsewhere. Root cause is the walker, not the LLM's SQL, which is semantically correct. |
| 963 | medicine_enzyme_interaction | **Other** | Same pattern as 3128 — `COUNT(mei.enzyme_id)` instead of `COUNT(*)`, one extra real non-gold column. |

**No category reaches the >=3 threshold**, even before this correction (the
raw classifier topped out at 2/5 for "Excess join column", itself
explicitly excluded as expected/non-fixable). After hand-verification, the
4 real failures split 2 "Other" / 1 "Missed a required table" / 1 "Missed a
required column" — no repeated, general, fixable prompt issue. The one
pattern that *does* repeat (2/4: `COUNT(specific_column)` vs `COUNT(*)`) is
below the locked >=3 bar and is noted for future reference, not acted on
now.

**Decision: lock `backward_v1`.** No `backward_v2` drafted — the decision
rule's threshold isn't met, and the failures that do occur are either a
model reasoning gap (3641, mirrors Method C's own worst-5 finding on the
*same* qid) or a walker-side limitation already known and accepted (5877),
not prompt-fixable patterns.


## 11. Final report

**Locked prompt version: `backward_v1`.** Run for real on the same 20
Spider train examples as the forward phase (`data/processed/prompt_iteration_set.json`,
seed=42), scored against sqlglot Tier-2 train gold. 15/20 (75%) perfect
`mean_f1`; 4/20 genuinely imperfect, no dominant fixable category (see
§10) — `backward_v2` was not drafted.

**Failure category counts** (4 real failures — the nominal 5th "worst"
query, qid 235, is a `mean_f1=1.0` tie-break artifact, not a failure; see
§10):

| Category | Count | qids |
| --- | --- | --- |
| Other | 2 | 3128, 963 |
| Missed a required table | 1 | 3641 |
| Missed a required column | 1 | 5877 |

**Total cost spent on this iteration phase:** see the cell below — summed
directly from `outputs/logs/llm_calls_prompt_iteration.jsonl`, filtered to
`metadata.phase == "backward_prompt_iteration"`. Well under the
`cost_cap_usd = cumulative_cap + 3 ~= $3.25` cap.

**Bonus comparison — does D's predicted table set intersect with C's, on
the same 20 questions?** Yes, on **20/20 (100%)**. On 18/20, C's and D's
table sets are byte-identical. The two exceptions:
- qid 1550: C additionally predicts `Customers` (D: `Accounts` only) —
  matches the forward phase's own finding that this qid's `Customers` is unused but
  topically plausible.
- qid 3385: C additionally predicts `competition_result` and `club` (D:
  `competition` only) — this is the *exact* over-inclusion pattern the forward phase
  identified as `forward_v1`'s dominant worst-5 failure ("Included
  join-only column"), reproduced again here, on the same qid, against the
  same gold. D does not exhibit this pattern on either of C's two
  extra-table cases — a preview that Method E's union will likely inherit
  C's over-inclusion tendency on these specific queries rather than being
  corrected by D.


In [21]:
import json
from pathlib import Path

_repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_log_path = _repo_root / "outputs" / "logs" / "llm_calls_prompt_iteration.jsonl"

phase_cost = 0.0
phase_calls = 0
grand_total_cost = 0.0
grand_total_calls = 0
with _log_path.open() as f:
    for line in f:
        entry = json.loads(line)
        grand_total_cost += entry["cost_usd"]
        grand_total_calls += 1
        if entry["metadata"].get("phase") == "backward_prompt_iteration":
            phase_cost += entry["cost_usd"]
            phase_calls += 1

print(f"Calls tagged phase=backward_prompt_iteration: {phase_calls}")
print(f"Cost this phase: ${phase_cost:.5f}")
print(f"Grand total in shared log (all phases): {grand_total_calls} calls, ${grand_total_cost:.5f}")


Calls tagged phase=backward_prompt_iteration: 80
Cost this phase: $0.16410
Grand total in shared log (all phases): 200 calls, $0.41611
